# Notebook PRO: Agente Conversacional Híbrido con LLM + RAG + Memoria
Este notebook simula un agente conversacional profesional, integrando:
- NLU básico
- RAG con ChromaDB
- OpenAI LLM
- Memoria conversacional
- Control de tokens y coste

##  Instalación de librerías necesarias

In [ ]:

!pip install openai chromadb tiktoken langchain --quiet


## Importación de librerías

In [ ]:

import openai
import chromadb
from chromadb.utils import embedding_functions
from langchain.text_splitter import CharacterTextSplitter
import os
import tiktoken

import os
os.environ["OPENAI_API_KEY"] = "poner aqui tu clave"


# Configuración inicial
openai.api_key = os.environ["OPENAI_API_KEY"]

## Indexación de documentos para RAG (simulados)

In [ ]:

documentos = [
    { "title": "Horario del curso", "content": "El curso de IA se imparte los lunes y miércoles de 17:00 a 19:00." },
    { "title": "Ubicación", "content": "Las clases se realizan en el aula 3.1 del edificio principal." },
    { "title": "Profesorado", "content": "El profesor del curso es el Dr. Sinesio David Carvajal Tabasco." }
]

# Inicializamos Chroma
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="curso_ia")

# Insertamos documentos
for i, doc in enumerate(documentos):
    collection.add(
        documents=[doc["content"]],
        metadatas=[{"title": doc["title"]}],
        ids=[f"doc_{i}"]
    )


## Función de conversación con recuperación de contexto + LLM

In [ ]:
import openai
import chromadb
import tiktoken

history = []

def contar_tokens(texto):
    enc = tiktoken.encoding_for_model("gpt-3.5-turbo")
    return len(enc.encode(texto))

def agente_ia_conversacional(prompt_usuario):
    global history

    # Paso 1: Recuperar contexto
    resultados = collection.query(query_texts=[prompt_usuario], n_results=1)
    contexto = resultados["documents"][0][0] if resultados["documents"] else ""

    # Paso 2: Preparar contexto conversacional
    prompt_base = f"""
Eres un asistente inteligente del curso universitario de Inteligencia Artificial.
Basándote en el siguiente contexto documental y en la conversación previa, responde de forma clara y útil.

Contexto recuperado:
{contexto}

Conversación previa:
{''.join(history[-4:])}

Pregunta:
{prompt_usuario}
"""

    # Paso 3: Llamada al LLM
    client = openai.OpenAI() # Create an OpenAI client instance
    response = client.chat.completions.create( # Use the new method
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "Asistente experto en IA conversacional educativa"},
            {"role": "user", "content": prompt_base}
        ],
        max_tokens=300,
        temperature=0.5
    )

    respuesta = response.choices[0].message.content
    history.append(f"Usuario: {prompt_usuario}\nAsistente: {respuesta}\n")

    # The usage information is now accessed differently
    tokens_usados = response.usage.total_tokens if response.usage else 0
    print(f"🧠 Tokens usados: {tokens_usados}")
    return respuesta

## Prueba inicial del agente

In [ ]:

pregunta = "¿Dónde son las clases?"
respuesta = agente_ia_conversacional(pregunta)
print(respuesta)


🧠 Tokens usados: 158
🤖 Las clases del curso de Inteligencia Artificial se realizan en el aula 3.1 del edificio principal. ¡Espero que esta información te sea de utilidad! ¿Hay algo más en lo que pueda ayudarte?
